# Aura AI — Continue Fine-Tuning Existing Korean Chat Checkpoint

Continue the existing Aura model from its best chat-SFT checkpoint. The architecture and tokenizer stay unchanged; this run adds supervised training steps and retains the original checkpoint unless validation improves.


In [ ]:
from pathlib import Path
import os, sys, zipfile, shutil, json, subprocess, hashlib, torch

PROJECT = Path("/kaggle/working/Aura-AI")
PROJECT.mkdir(parents=True, exist_ok=True)

input_root = Path("/kaggle/input")
print("Mounted input roots:", [p.name for p in input_root.iterdir()], flush=True)
source_candidates = [p for p in input_root.rglob("prepare_korean_sft.py") if p.parent.name == "scripts"]
assert source_candidates, "Aura SFT v2 source files are not mounted"
source_root = source_candidates[0].parent.parent
shutil.copytree(source_root, PROJECT, dirs_exist_ok=True)

# Resume from the existing Aura chat-SFT checkpoint; keep the same model and tokenizer.
train_script = PROJECT / "scripts" / "train_chat_sft.py"
train_code = train_script.read_text(encoding="utf-8")
patches = [
    ('best_loss = float("inf")\n    best_step = 0',
     'source_step = int(saved.get("step", 0))\n    best_loss = evaluate(model, valid_loader, "cuda", dtype, args.eval_batches)\n    best_step = source_step\n    print(json.dumps({"baseline_validation_loss": best_loss, "source_step": source_step}), flush=True)'),
    ('completed_step = 0', 'completed_step = source_step'),
    ('        completed_step = step', '        global_step = source_step + step\n        completed_step = global_step'),
    ('                best_step = step', '                best_step = global_step'),
    ('best_path, model, step, val_loss, args, source,', 'best_path, model, global_step, val_loss, args, source,'),
    ('"step": step,', '"step": global_step,'),
    ('latest_path, model, step, val_loss, args, source,', 'latest_path, model, global_step, val_loss, args, source,'),
    ('{"early_stopping": True, "step": step, "best_step": best_step}', '{"early_stopping": True, "step": global_step, "best_step": best_step}')
]
for old, new in patches:
    if old not in train_code:
        raise RuntimeError("Continuation patch anchor missing: " + old[:80])
    train_code = train_code.replace(old, new, 1)
train_script.write_text(train_code, encoding="utf-8")

tokenizer_candidates = list(input_root.rglob("aura-bpe.json"))
assert tokenizer_candidates, "Paired Aura tokenizer missing from Kaggle inputs"
tokenizer_path = PROJECT / "data" / "tokenizer" / "aura-bpe.json"
tokenizer_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(tokenizer_candidates[0], tokenizer_path)

sft_candidates = list(input_root.rglob("best_chat_sft.pt"))
assert sft_candidates, "Existing Aura best_chat_sft.pt checkpoint missing from Kaggle inputs"
source_checkpoint_path = PROJECT / "data" / "checkpoints" / "best_chat_sft.pt"
source_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(sft_candidates[0], source_checkpoint_path)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
assert torch.cuda.is_available(), "Enable the Kaggle GPU accelerator"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

from aura.config import MODEL, TOKENIZER_PATH
from aura.tokenizer import AuraTokenizer

tokenizer = AuraTokenizer.load(TOKENIZER_PATH)
assert tokenizer.vocab_size == MODEL.vocab_size
checkpoint = torch.load(source_checkpoint_path, map_location="cpu", weights_only=False)
tokenizer_sha = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
assert checkpoint.get("tokenizer_sha256") == tokenizer_sha, "Pretrain checkpoint/tokenizer SHA-256 mismatch"
print({
    "gpu": torch.cuda.get_device_name(0),
    "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "existing_sft_step": checkpoint.get("step"),\n    "existing_validation_loss": checkpoint.get("validation_loss"),
    "tokenizer_vocab": tokenizer.vocab_size,
    "checkpoint_bytes": source_checkpoint_path.stat().st_size,
    "tokenizer_sha256": tokenizer_sha,
}, flush=True)
assert checkpoint.get("stage") == "chat_sft" and checkpoint.get("step", 0) >= 3900, "Expected the existing Aura chat-SFT checkpoint"


In [ ]:
from scripts.prepare_korean_sft import main as prepare_korean_sft
prepare_korean_sft()
report = json.loads((PROJECT / "data" / "prepared" / "korean_sft_v2_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)


In [ ]:
from scripts.train_chat_sft import main as train_chat_sft
sys.argv = [
    "train_chat_sft",
    "--source", str(PROJECT / "data" / "checkpoints" / "best_chat_sft.pt"),
    "--steps", "2000",
    "--batch-size", "2",
    "--grad-accum", "4",
    "--seq-len", "512",
    "--lr", "0.000002",
    "--weight-decay", "0.05",
    "--dropout", "0.05",
    "--warmup-steps", "100",
    "--eval-every", "100",
    "--eval-batches", "128",
    "--patience", "8",
    "--min-delta", "0.001",
    "--checkpoint-every", "250",
]
train_chat_sft()


In [ ]:
from aura.models.transformer import AuraModel
from scripts.chat_local import make_prompt
from aura.tokenizer import EOS

chat_checkpoint_path = PROJECT / "data" / "checkpoints" / "latest_chat_sft.pt"
chat_checkpoint = torch.load(chat_checkpoint_path, map_location="cpu", weights_only=False)
model = AuraModel(MODEL).cuda()
model.load_state_dict(chat_checkpoint["model"], strict=True)
model.eval()

test_prompts = [
    "안녕. 너는 어떤 AI야?",
    "파이썬 리스트와 튜플의 차이를 예시와 함께 설명해줘.",
    "검증 손실이 계속 증가하면 학습을 어떻게 조정해야 해?",
]
for prompt in test_prompts:
    ids = make_prompt(tokenizer, [("user", prompt)])
    input_ids = torch.tensor([ids], dtype=torch.long, device="cuda")
    with torch.inference_mode():
        output = model.generate(input_ids, max_new_tokens=96, temperature=0.7, top_k=30)
    generated = output[0, len(ids):].tolist()
    if EOS in generated:
        generated = generated[:generated.index(EOS)]
    print("\nUSER:", prompt)
    print("AURA:", tokenizer.decode(generated).strip())

print({
    "checkpoint_step": chat_checkpoint.get("step"),
    "best_validation_loss": chat_checkpoint.get("validation_loss"),
    "generation_smoke_test": "completed",
}, flush=True)


In [ ]:
out = Path("/kaggle/working/aura-korean-sft-v2-artifacts")
out.mkdir(parents=True, exist_ok=True)
export_paths = [
    "data/checkpoints/latest_chat_sft.pt",
    "data/checkpoints/best_chat_sft.pt",
    "data/prepared/chat_sft_v2_log.jsonl",
    "data/prepared/korean_sft_v2_manifest.json",
    "data/chat/train.jsonl",
    "data/chat/valid.jsonl",
    "data/tokenizer/aura-bpe.json",
    "scripts/chat_local.py",
    "scripts/train_chat_sft.py",
    "scripts/prepare_korean_sft.py",
]
for relative in export_paths:
    source = PROJECT / relative
    assert source.is_file(), f"Expected artifact missing: {source}"
    target = out / relative.replace("/", "__")
    shutil.copy2(source, target)
    print("EXPORTED", relative, target.stat().st_size, flush=True)

zip_path = shutil.make_archive("/kaggle/working/aura-korean-sft-v2-artifacts", "zip", out)
print("AURA_KOREAN_SFT_V2_ARTIFACTS_READY", zip_path, Path(zip_path).stat().st_size, flush=True)
